# Tính Compression Ratio cho 4 hệ thống trên 200 mẫu LLM-as-a-Judge

Notebook này ghép `llm_judge_sample_200.jsonl` với bốn file prediction đầy đủ của **Lead-1, Lead-3, Scratch Transformer và ViT5** bằng trường `id`, sau đó xuất:

`/kaggle/working/llm_judge_sample_200_eval.jsonl`

Công thức sử dụng:

$$\text{compression ratio} = \frac{\text{số từ của prediction}}{\text{số từ của source}}$$

Trong file output:

- File chỉ có **một dòng JSON**, chứa Compression Ratio trung bình trên 200 mẫu của bốn hệ thống.
- Không ghi kết quả theo từng mẫu, `id`, `source`, `reference` hoặc `prediction`.
- Giá trị được lưu ở dạng số thập phân, ví dụ `0.125` tương ứng với `12.5%`.
- Số từ được đếm sau khi chuẩn hóa Unicode NFC và tách theo khoảng trắng (`text.split()`).

> Bạn chỉ cần sửa **5 đường dẫn** trong cell Cấu hình rồi chọn **Run all**. Không cần cài thêm thư viện.

## 1. Cấu hình đường dẫn

In [ ]:
from pathlib import Path
from collections import Counter
from statistics import mean
try:
    from IPython.display import FileLink, display
except ImportError:
    def display(value):
        print(value)

    def FileLink(path, result_html_prefix=""):
        return f"{result_html_prefix}{path}"
import hashlib
import json
import os
import unicodedata

# ============================================================
# CHỈ CẦN SỬA 5 ĐƯỜNG DẪN DƯỚI ĐÂY
# ============================================================
LLM_JUDGE_SAMPLE_PATH = Path(
    os.environ.get(
        "CR_LLM_JUDGE_SAMPLE_PATH",
        "/kaggle/input/your-dataset/llm_judge_sample_200.jsonl",
    )
)

LEAD1_PREDICTION_PATH = Path(
    os.environ.get(
        "CR_LEAD1_PREDICTION_PATH",
        "/kaggle/input/your-dataset/lead1.jsonl",
    )
)

LEAD3_PREDICTION_PATH = Path(
    os.environ.get(
        "CR_LEAD3_PREDICTION_PATH",
        "/kaggle/input/your-dataset/lead3.jsonl",
    )
)

TRANSFORMER_PREDICTION_PATH = Path(
    os.environ.get(
        "CR_TRANSFORMER_PREDICTION_PATH",
        "/kaggle/input/your-dataset/scratch_transformer_test_core_2000.jsonl",
    )
)

VIT5_PREDICTION_PATH = Path(
    os.environ.get(
        "CR_VIT5_PREDICTION_PATH",
        "/kaggle/input/your-dataset/vit5_test_core_2000.jsonl",
    )
)

# Không cần sửa: file luôn được lưu trong /kaggle/working khi chạy trên Kaggle.
OUTPUT_PATH = Path(
    os.environ.get(
        "CR_OUTPUT_PATH",
        "/kaggle/working/llm_judge_sample_200_eval.jsonl",
    )
)

EXPECTED_SAMPLE_COUNT = int(os.environ.get("CR_EXPECTED_SAMPLE_COUNT", "200"))
ROUND_DIGITS = 6

PREDICTION_PATHS = {
    "lead1": LEAD1_PREDICTION_PATH,
    "lead3": LEAD3_PREDICTION_PATH,
    "transformer": TRANSFORMER_PREDICTION_PATH,
    "vit5": VIT5_PREDICTION_PATH,
}

print("Đã thiết lập đường dẫn. Output:", OUTPUT_PATH)

## 2. Hàm đọc và kiểm tra JSONL

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    """Đọc JSONL, báo rõ số dòng nếu JSON không hợp lệ."""
    if not path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy file: {path}\n"
            "Hãy kiểm tra lại đường dẫn trong cell Cấu hình."
        )

    rows = []
    with path.open("r", encoding="utf-8-sig") as file:
        for line_number, raw_line in enumerate(file, start=1):
            line = raw_line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"JSON không hợp lệ tại {path}, dòng {line_number}: {exc}"
                ) from exc
            if not isinstance(row, dict):
                raise ValueError(
                    f"Mỗi dòng phải là một JSON object: {path}, dòng {line_number}."
                )
            rows.append(row)

    if not rows:
        raise ValueError(f"File không có dữ liệu: {path}")
    return rows


def require_fields(row: dict, fields: tuple[str, ...], context: str) -> None:
    missing = [field for field in fields if field not in row]
    if missing:
        raise ValueError(f"{context} thiếu trường bắt buộc: {missing}")


def build_unique_id_index(rows: list[dict], path: Path) -> dict[str, dict]:
    """Lập chỉ mục theo id và không cho phép id rỗng/trùng."""
    ids = []
    index = {}

    for row_number, row in enumerate(rows, start=1):
        require_fields(row, ("id",), f"{path}, dòng dữ liệu {row_number}")
        sample_id = str(row["id"]).strip()
        if not sample_id:
            raise ValueError(f"{path}, dòng dữ liệu {row_number}: id rỗng.")
        ids.append(sample_id)
        index[sample_id] = row

    duplicate_ids = sorted(
        sample_id for sample_id, count in Counter(ids).items() if count > 1
    )
    if duplicate_ids:
        preview = duplicate_ids[:10]
        raise ValueError(
            f"{path} có {len(duplicate_ids)} id trùng. Ví dụ: {preview}"
        )

    return index


def validate_sample_rows(sample_rows: list[dict]) -> list[str]:
    if len(sample_rows) != EXPECTED_SAMPLE_COUNT:
        raise ValueError(
            f"llm_judge_sample phải có đúng {EXPECTED_SAMPLE_COUNT} mẫu, "
            f"nhưng đọc được {len(sample_rows)} mẫu."
        )

    build_unique_id_index(sample_rows, LLM_JUDGE_SAMPLE_PATH)
    ordered_ids = []

    for row_number, row in enumerate(sample_rows, start=1):
        require_fields(
            row,
            ("id", "source"),
            f"{LLM_JUDGE_SAMPLE_PATH}, dòng dữ liệu {row_number}",
        )
        sample_id = str(row["id"]).strip()
        source = row["source"]
        if not isinstance(source, str) or not source.strip():
            raise ValueError(f"Mẫu {sample_id} có source rỗng hoặc không phải chuỗi.")
        ordered_ids.append(sample_id)

    return ordered_ids


def select_predictions_for_sample(
    path: Path, system_key: str, ordered_ids: list[str]
) -> dict[str, dict]:
    """Đọc file prediction 2.000 mẫu và lấy đúng 200 id cần đánh giá."""
    rows = read_jsonl(path)
    full_index = build_unique_id_index(rows, path)

    missing_ids = [sample_id for sample_id in ordered_ids if sample_id not in full_index]
    if missing_ids:
        raise ValueError(
            f"File {system_key} thiếu {len(missing_ids)}/{len(ordered_ids)} id cần đánh giá. "
            f"Ví dụ: {missing_ids[:10]}"
        )

    selected = {}
    for sample_id in ordered_ids:
        row = full_index[sample_id]
        require_fields(row, ("prediction",), f"{system_key}, id={sample_id}")

        status = str(row.get("status", "ok")).strip().lower()
        if status != "ok":
            raise ValueError(
                f"{system_key}, id={sample_id} có status={status!r}; "
                f"error={row.get('error')!r}."
            )

        prediction = row["prediction"]
        if not isinstance(prediction, str) or not prediction.strip():
            raise ValueError(
                f"{system_key}, id={sample_id}: prediction rỗng hoặc không phải chuỗi."
            )
        selected[sample_id] = row

    print(
        f"✓ {system_key:<11}: đọc {len(rows):>4} prediction, "
        f"lấy đúng {len(selected)} mẫu theo llm_judge_sample."
    )
    return selected

## 3. Đọc dữ liệu và ghép đúng 200 ID

In [ ]:
sample_rows = read_jsonl(LLM_JUDGE_SAMPLE_PATH)
ordered_ids = validate_sample_rows(sample_rows)

print(f"✓ llm_judge_sample: {len(sample_rows)} mẫu, id không trùng, source không rỗng.")

predictions_by_system = {
    system_key: select_predictions_for_sample(path, system_key, ordered_ids)
    for system_key, path in PREDICTION_PATHS.items()
}

print(f"\nTất cả bốn file prediction đều bao phủ đủ {EXPECTED_SAMPLE_COUNT} ID.")

## 4. Tính Compression Ratio

Notebook tính Compression Ratio riêng cho từng mẫu trong bộ nhớ, sau đó lấy **trung bình cộng trên 200 mẫu** cho từng hệ thống. File output chỉ lưu bốn giá trị trung bình cuối cùng.

In [ ]:
def normalize_text(text: str) -> str:
    return unicodedata.normalize("NFC", text).strip()


def count_whitespace_words(text: str) -> int:
    """Đếm đơn vị ngăn cách bởi khoảng trắng sau chuẩn hóa Unicode NFC."""
    return len(normalize_text(text).split())


def calculate_compression_ratio(summary: str, source: str) -> tuple[int, int, float]:
    source_word_count = count_whitespace_words(source)
    summary_word_count = count_whitespace_words(summary)

    if source_word_count == 0:
        raise ValueError("Không thể tính compression ratio vì source có 0 từ.")

    ratio = summary_word_count / source_word_count
    return source_word_count, summary_word_count, ratio


SYSTEM_KEYS = ("lead1", "lead3", "transformer", "vit5")
sample_by_id = {str(row["id"]).strip(): row for row in sample_rows}
ratios_by_system = {system_key: [] for system_key in SYSTEM_KEYS}

for sample_id in ordered_ids:
    sample = sample_by_id[sample_id]
    source = sample["source"]

    for system_key in SYSTEM_KEYS:
        prediction_row = predictions_by_system[system_key][sample_id]
        prediction = prediction_row["prediction"]
        _, _, ratio = calculate_compression_ratio(prediction, source)
        ratios_by_system[system_key].append(ratio)

for system_key, ratios in ratios_by_system.items():
    if len(ratios) != EXPECTED_SAMPLE_COUNT:
        raise RuntimeError(
            f"{system_key} phải có {EXPECTED_SAMPLE_COUNT} tỷ lệ, nhưng có {len(ratios)}."
        )

mean_compression_ratios = {
    system_key: round(mean(ratios_by_system[system_key]), ROUND_DIGITS)
    for system_key in SYSTEM_KEYS
}

print(
    f"✓ Đã tính Compression Ratio trên {EXPECTED_SAMPLE_COUNT} mẫu "
    "và lấy trung bình riêng cho 4 hệ thống."
)

## 5. Xem kết quả trung bình

Bảng dưới đây chính là bốn giá trị sẽ được ghi vào file output.

In [ ]:
summary_rows = [
    {
        "system": system_key,
        "mean_compression_ratio": mean_compression_ratios[system_key],
    }
    for system_key in SYSTEM_KEYS
]

try:
    import pandas as pd
    display(pd.DataFrame(summary_rows))
except ImportError:
    for row in summary_rows:
        print(row)

print("\nNội dung sẽ ghi vào output:")
print(json.dumps(mean_compression_ratios, ensure_ascii=False, indent=2))

## 6. Xuất `llm_judge_sample_200_eval.jsonl`

Cell này ghi một dòng JSON chứa bốn giá trị trung bình, sau đó đọc lại để xác minh và in SHA-256.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
temporary_path = OUTPUT_PATH.with_suffix(OUTPUT_PATH.suffix + ".tmp")

with temporary_path.open("w", encoding="utf-8", newline="\n") as file:
    file.write(json.dumps(mean_compression_ratios, ensure_ascii=False) + "\n")

temporary_path.replace(OUTPUT_PATH)

# Đọc lại file vừa xuất để phát hiện lỗi ghi file hoặc lỗi JSONL.
verified_rows = read_jsonl(OUTPUT_PATH)
if len(verified_rows) != 1:
    raise RuntimeError(f"Output phải có đúng 1 dòng, nhưng có {len(verified_rows)} dòng.")

if verified_rows[0] != mean_compression_ratios:
    raise RuntimeError("Nội dung đọc lại không khớp bốn giá trị trung bình đã tính.")

sha256 = hashlib.sha256(OUTPUT_PATH.read_bytes()).hexdigest()
file_size_kb = OUTPUT_PATH.stat().st_size / 1024

print("✓ Xuất file thành công")
print("  Đường dẫn :", OUTPUT_PATH)
print("  Số dòng   :", len(verified_rows))
print(f"  Kích thước: {file_size_kb:,.2f} KB")
print("  SHA-256    :", sha256)

display(FileLink(str(OUTPUT_PATH), result_html_prefix="Tải file kết quả: "))

## Schema của file output

```json
{
  "lead1": 0.083333,
  "lead3": 0.152381,
  "transformer": 0.107619,
  "vit5": 0.095238
}
```